# 01. Raw Overview — 원본 카탈로그

이 노트북은 원본 테이블의 구조·기간·집계 행과 최소 품질 점검만 기록한다. 지연 기준, 시간차 해석, 지연사유 표준화, 목적지·국제선 분류는 `02_preprocessing_views` 또는 이후 단계에서 다룬다.

모든 조회는 읽기 전용이며 `raw_*` 테이블과 원본 파일을 수정하지 않는다.

## 0. 연결 설정

In [1]:
from pathlib import Path
import os
import re

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

load_dotenv(ROOT / '.env')
SQL_FILE = ROOT / 'sql' / '01_raw_overview.sql'

db_config = {
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT', '3306')),
    'database': os.getenv('DB_NAME'),
    'username': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
}
missing = [key for key in ['host', 'database', 'username', 'password'] if not db_config[key]]
if missing:
    raise ValueError(f'.env의 DB 설정이 비어 있음: {missing}')

engine = create_engine(URL.create('mysql+pymysql', **db_config), pool_pre_ping=True, future=True)

def load_named_queries(path: Path) -> dict[str, str]:
    sql_text = path.read_text(encoding='utf-8')
    matches = list(re.finditer(r'^-- name: ([a-z0-9_]+)\s*$', sql_text, flags=re.MULTILINE))
    return {
        match.group(1): sql_text[match.end(): matches[index + 1].start() if index + 1 < len(matches) else len(sql_text)].strip().rstrip(';')
        for index, match in enumerate(matches)
    }

SQL = load_named_queries(SQL_FILE)

def query(name: str) -> pd.DataFrame:
    return pd.read_sql(text(SQL[name]), engine)

with engine.connect() as connection:
    connection.execute(text('SELECT 1'))

print(f"Connected to MySQL database: {db_config['database']}")

Connected to MySQL database: korea_outbound_travel


## 1. 테이블 범위·스키마

In [2]:
# 원본 테이블별 행 수와 전체 컬럼 구조를 함께 확인한다.
table_counts = query('table_counts')
table_columns = query('table_columns')
table_shape = table_counts.merge(
    table_columns.groupby('raw_table_name', as_index=False)['raw_column_name'].count().rename(columns={'raw_column_name': 'column_count'}),
    left_on='table_name', right_on='raw_table_name', how='left',
).drop(columns='raw_table_name')
display(table_shape.rename(columns={'table_name': '테이블', 'row_count': '행 수', 'column_count': '열 수'}))
display(table_columns.rename(columns={
    'raw_table_name': '원본 테이블', 'column_order': '순서', 'raw_column_name': '컬럼명',
    'mysql_type': 'MySQL 타입', 'null_allowed': 'NULL 허용',
}))

,테이블,행 수,열 수
0,raw_route_passenger,9670,4
1,raw_route_operations,9670,4
2,raw_airline_passenger,3951,3
3,raw_airline_operations,3951,3
4,raw_airport_passenger,301,3
5,raw_country_passenger,2083,4
6,raw_departure_flight,793865,12


,원본 테이블,순서,컬럼명,MySQL 타입,NULL 허용
0,raw_airline_operations,1,airline_name_raw,varchar(150),NO
1,raw_airline_operations,2,period_raw,varchar(20),NO
2,raw_airline_operations,3,operations,bigint,YES
3,raw_airline_passenger,1,airline_name_raw,varchar(150),NO
4,raw_airline_passenger,2,period_raw,varchar(20),NO
5,raw_airline_passenger,3,passengers,bigint,YES
6,raw_airport_passenger,1,airport_name_raw,varchar(100),NO
7,raw_airport_passenger,2,period_raw,varchar(20),NO
8,raw_airport_passenger,3,passengers,bigint,YES
9,raw_country_passenger,1,region_raw,varchar(100),NO


## 2. 월간 집계 원본 카탈로그

In [3]:
# 기간·수치 결측·0값과 월별 적재 범위를 확인한다.
display(query('aggregate_quality_summary'))
display(query('period_values'))

,dataset,rows_count,first_period,last_period,period_count,metric_nulls,metric_zeros
0,route_passenger,9670,2023년 01월,2025년 12월,36,0.0,1107.0
1,route_operations,9670,2023년 01월,2025년 12월,36,0.0,241.0
2,airline_passenger,3951,2023년 01월,2025년 12월,36,0.0,803.0
3,airline_operations,3951,2023년 01월,2025년 12월,36,0.0,0.0
4,airport_passenger,301,2023년 01월,2025년 12월,36,0.0,0.0
5,country_passenger,2083,2023년 01월,2025년 12월,36,0.0,289.0


,dataset,period_raw,rows_count
0,airline_operations,2023년 01월,101
1,airline_operations,2023년 02월,101
2,airline_operations,2023년 03월,105
3,airline_operations,2023년 04월,103
4,airline_operations,2023년 05월,102
...,...,...,...
211,route_passenger,2025년 08월,301
212,route_passenger,2025년 09월,301
213,route_passenger,2025년 10월,315
214,route_passenger,2025년 11월,287


In [4]:
# 집계 행 표기와 원본 합계 대비 세부 행 합계의 완전성을 확인한다.
display(query('aggregate_row_profile'))
display(query('aggregate_row_completeness_check'))

# NULL·소계 후보는 자동 제외하지 않고 원문값을 확인한다.
display(query('aggregate_label_discovery'))

,table_name,aggregate_value,rows_count
0,raw_airline_operations,국적사 계,36
1,raw_airline_operations,외항사 계,36
2,raw_airline_operations,전체 합계,36
3,raw_airline_passenger,국적사 계,36
4,raw_airline_passenger,외항사 계,36
5,raw_airline_passenger,전체 합계,36
6,raw_airport_passenger,전체 합계,36
7,raw_country_passenger,전체 합계,36
8,raw_route_operations,전체 합계,36
9,raw_route_passenger,전체 합계,36


,dataset,month_count,reported_total_missing_months,total_mismatch_months,subtotal_mismatch_months,min_parts_difference,max_parts_difference,completeness_check
0,airline_operations,36,0.0,0.0,0.0,0.0,0.0,matched
1,airline_passenger,36,0.0,0.0,0.0,0.0,0.0,matched
2,airport_passenger,36,0.0,0.0,0.0,0.0,0.0,matched
3,country_passenger,36,0.0,0.0,0.0,0.0,0.0,matched
4,route_operations,36,0.0,0.0,0.0,0.0,0.0,matched
5,route_passenger,36,0.0,0.0,0.0,0.0,0.0,matched


,field_name,raw_value,rows_count
0,airline_operations_name,국적사 계,36
1,airline_operations_name,외항사 계,36
2,airline_operations_name,전체 합계,36
3,airline_passenger_name,국적사 계,36
4,airline_passenger_name,외항사 계,36
5,airline_passenger_name,전체 합계,36
6,airport_name,전체 합계,36
7,country_name,None,36
8,country_region,전체 합계,36
9,route_operations_destination,장가계(대용)(DYG),144


## 3. 출발 로그 적재·원본 품질

In [5]:
# 편명 형식과 공항·일자·편명·계획시간 기준 중복 후보를 확인한다.
display(query('flight_number_profile'))
display(query('flight_duplicate_candidates'))

,flight_number_format,rows_count,distinct_flight_number_count,example_flight_numbers
0,two_letter_number,677004,2623,"AA127, AA128, AA280, AA9600, AA9601, AA9602, A..."
1,three_letter_number,149,102,"ABD4004, ABD4008, ABD4977, ADB3031, ADB3050, A..."
2,other,116712,1604,"3CMAG, 3S504, 3S504A, 3S562, 3S563, 3S564, 3S5..."


,result_type,airport_name_raw,flight_date_raw,flight_number_raw,scheduled_time_raw,duplicate_rows,duplicate_group_count,duplicate_row_count
0,summary,None,None,None,None,NaN,150.0,300.0
1,example,김포,20230326,OZ8915,07:55,2.0,NaN,NaN
2,example,김포,20230326,OZ8943,11:50,2.0,NaN,NaN
3,example,김포,20230326,OZ8967,16:35,2.0,NaN,NaN
4,example,김포,20230327,OZ8915,07:55,2.0,NaN,NaN
5,example,김포,20230327,OZ8943,11:50,2.0,NaN,NaN
6,example,김포,20230327,OZ8967,16:35,2.0,NaN,NaN
7,example,김포,20231029,LJ633,18:55,2.0,NaN,NaN
8,example,김포,20240331,ZE203,06:15,2.0,NaN,NaN
9,example,김포,20240331,ZE215,12:55,2.0,NaN,NaN


In [6]:
# 일자 형식 검증 후 공항별 적재 기간과 방향·서비스·상태 원문값을 확인한다.
display(query('flight_date_profile'))
display(query('log_monthly_overview'))
display(query('log_status_service'))

,airport_name_raw,total_rows,date_null_or_blank,date_format_invalid,date_not_a_real_date,first_date,last_date,distinct_date_count
0,김포,199127,0.0,0.0,0.0,20230101,20251231,1096
1,인천,594738,0.0,0.0,0.0,20230101,20251231,1096


,airport_name_raw,operating_month,rows_count
0,김포,202301,5645
1,인천,202301,11628
2,김포,202302,5100
3,인천,202302,10867
4,김포,202303,5771
...,...,...,...
67,인천,202510,18632
68,김포,202511,5637
69,인천,202511,17405
70,김포,202512,5987


,airport_name_raw,direction_raw,service_type_raw,status_raw,rows_count
0,김포,출발,여객,출발,156485
1,김포,출발,여객,지연,40346
2,김포,출발,여객,취소,1379
3,김포,출발,여객,None,693
4,김포,출발,여객,회항,83
5,김포,출발,기타,출발,80
6,김포,출발,기타,None,32
7,김포,출발,기타,지연,22
8,김포,출발,기타,취소,3
9,김포,출발,화물,지연,3


In [7]:
# 시간값 결측·':' 플레이스홀더·형식 오류와 자릿수 패턴을 확인한다.
display(query('log_quality_strict'))
display(query('invalid_time_pattern_profile'))

# 오류값은 자동 복원하지 않고 원문 사례를 남긴다.
display(query('invalid_time_samples'))

,airport_name_raw,total_rows,scheduled_null_or_blank,scheduled_colon_placeholder,scheduled_invalid_clock,scheduled_unusable,estimated_null_or_blank,estimated_colon_placeholder,estimated_invalid_clock,estimated_unusable,actual_null_or_blank,actual_colon_placeholder,actual_invalid_clock,actual_unusable
0,김포,199127,0.0,0.0,516.0,516.0,0.0,1562.0,505.0,2067.0,0.0,2232.0,494.0,2726.0
1,인천,594738,0.0,0.0,1614.0,1614.0,0.0,6655.0,1521.0,8176.0,0.0,8785.0,1390.0,10175.0


,field_name,digit_length,numeric_only_rows,recoverable_as_h_mm_candidates,rows_count,distinct_value_count,examples
0,actual,1,16.0,0.0,16,8,"1:, 2:, 3:, 4:, 5:, 6:, 8:, 9:"
1,actual,2,92.0,0.0,92,40,"10:, 13:, 14:, 15:, 16:, 17:, 18:, 20:, 21:, 22:"
2,actual,3,1776.0,1776.0,1776,389,"10:2, 10:3, 10:4, 10:6, 10:8, 11:0, 11:2, 11:3..."
3,estimated,1,19.0,0.0,19,2,"0:, 5:"
4,estimated,2,82.0,0.0,82,11,"10:, 15:, 20:, 25:, 30:, 35:, 40:, 45:, 50:, 55:"
5,estimated,3,1924.0,1923.0,1924,110,"07:4, 10:0, 10:5, 11:0, 11:5, 12:0, 12:5, 12:9..."
6,estimated,4,1.0,0.0,1,1,13:68
7,scheduled,1,58.0,0.0,58,2,"0:, 5:"
8,scheduled,2,110.0,0.0,110,10,"10:, 15:, 20:, 25:, 30:, 35:, 40:, 45:, 50:, 55:"
9,scheduled,3,1962.0,1962.0,1962,96,"10:0, 10:5, 11:0, 11:5, 12:0, 12:5, 13:0, 13:5..."


,airport_name_raw,scheduled_time_raw,estimated_time_raw,actual_departure_time_raw,status_raw,destination_raw
0,김포,08:45,84:5,90:0,출발,제주
1,김포,60:0,60:0,61:3,출발,제주
2,김포,60:5,60:5,61:9,출발,제주
3,김포,62:0,62:0,63:1,출발,제주
4,김포,62:0,62:0,63:3,출발,제주
5,김포,63:0,63:0,63:7,출발,제주
6,김포,63:5,63:5,64:9,출발,제주
7,김포,64:0,64:0,65:5,출발,제주
8,김포,64:0,64:0,70:3,지연,제주
9,김포,64:5,64:5,70:0,출발,제주


## 4. 01 단계 요약

- 월간 집계 통계에는 분석 시 제외해야 할 집계 행(`전체 합계`, `국적사 계`, `외항사 계`)이 섞여 있으며, 원본 합계와 세부 행 합계를 별도로 대조한다.
- 출발 로그는 인천·김포의 2023~2025년 기록이며, 일자·서비스 구분·상태·시간 원문값의 품질 확인이 필요하다.
- 시간값에는 NULL·빈값뿐 아니라 `:` 플레이스홀더와 형식 오류가 존재하며, 오류값을 임의로 복원하지 않는다.
- 편명·계획시간 기준의 중복 후보가 있으나, 삭제 여부는 02 전처리 단계에서 판단한다.
- 01은 원본 카탈로그 단계이므로 지연 정의·국제선 판별·자정 보정 같은 분석 규칙은 여기서 확정하지 않는다.